# Blackstart City — GRPO from SFT Weights

Runs **GRPO reinforcement learning** starting from the pre-trained SFT adapter stored at
`SidditaVarma/Built-different` (HF Space, path `latest/blackstart-city-sft`).

**Runtime:** L4 / A100 recommended. 500 steps ≈ 35 min on L4.

---
### Steps
1. Install deps
2. Patch Unsloth dtype bugs (required every fresh runtime)
3. Pull SFT adapter + dataset from HF Space
4. Load model
5. Train with GRPO (5 reward signals, SFT warm-start)
6. Save + graphs + push to HF

In [ ]:
# ── 1. Install ────────────────────────────────────────────────────────────────
!pip install unsloth trl>=0.24.0 peft accelerate datasets huggingface_hub -q
!pip install git+https://github.com/Ankitpatil944/blackout-city.git -q

In [ ]:
# ── 2. C compiler (required by Triton) ───────────────────────────────────────
import subprocess, os, shutil

if not shutil.which("gcc"):
    subprocess.run(["apt-get", "install", "-y", "-q", "gcc"], check=True)
os.environ["CC"] = shutil.which("gcc") or "gcc"
print("CC =", os.environ["CC"])

In [ ]:
# ── 3. Patch Unsloth dtype bugs ───────────────────────────────────────────────
# Must run on every fresh runtime before loading any model.
# Fixes: Half/Float mismatches in matmul_lora, LoRA_MLP.backward, LoRA_W.backward

# ── utils.py: make matmul_lora use the autocast dtype, not raw tensor dtype ──
utils_path = "/usr/local/lib/python3.10/site-packages/unsloth/kernels/utils.py"
with open(utils_path) as f:
    src = f.read()

for old, new in [
    ("        dtype = out.dtype  # match 4-bit dequant output, not float32 activations\n",
     "        dtype = torch.get_autocast_gpu_dtype() if torch.is_autocast_enabled() else out.dtype\n        out   = out.to(dtype)\n"),
    ("        dtype = out.dtype\n",
     "        dtype = torch.get_autocast_gpu_dtype() if torch.is_autocast_enabled() else out.dtype\n        out   = out.to(dtype)\n"),
    ("        out   = out.to(X.dtype)\n        dtype = X.dtype\n",
     "        dtype = torch.get_autocast_gpu_dtype() if torch.is_autocast_enabled() else out.dtype\n        out   = out.to(dtype)\n"),
]:
    if old in src:
        src = src.replace(old, new, 1)
        print("utils.py  ✓ patched")
        break
else:
    if "torch.get_autocast_gpu_dtype()" in src:
        print("utils.py  ✓ already patched")
    else:
        print("utils.py  ✗ pattern not found — check manually")
with open(utils_path, "w") as f:
    f.write(src)

# ── fast_lora.py: fix LoRA_MLP.backward and LoRA_W.backward dtype handling ───
lora_path = "/usr/local/lib/python3.10/site-packages/unsloth/kernels/fast_lora.py"
with open(lora_path) as f:
    src = f.read()

patches = [
    # LoRA_MLP backward — define _cd from autocast dtype
    ("        dtype = X.dtype\n\n        gateA, gateB, upA, upB, downA, downB = (",
     "        _cd   = torch.get_autocast_gpu_dtype() if torch.is_autocast_enabled() else gateA.dtype\n        dtype = _cd\n        X     = X.to(_cd)\n        dY    = dY.to(_cd)\n\n        gateA, gateB, upA, upB, downA, downB = ("),
    ("        dtype = gateA.dtype\n        X     = X.to(dtype)\n        dY    = dY.to(dtype)\n",
     "        _cd   = torch.get_autocast_gpu_dtype() if torch.is_autocast_enabled() else gateA.dtype\n        dtype = _cd\n        X     = X.to(_cd)\n        dY    = dY.to(_cd)\n"),
    # LoRA_MLP backward — empty_like → explicit dtype
    ("        d_downA = torch.empty_like(downA)\n        d_downB = torch.empty_like(downB)\n        d_gateA = torch.empty_like(gateA)\n        d_gateB = torch.empty_like(gateB)\n        d_upA = torch.empty_like(upA)\n        d_upB = torch.empty_like(upB)\n",
     "        d_downA = torch.empty(downA.shape, dtype=_cd, device=downA.device)\n        d_downB = torch.empty(downB.shape, dtype=_cd, device=downB.device)\n        d_gateA = torch.empty(gateA.shape, dtype=_cd, device=gateA.device)\n        d_gateB = torch.empty(gateB.shape, dtype=_cd, device=gateB.device)\n        d_upA   = torch.empty(upA.shape,   dtype=_cd, device=upA.device)\n        d_upB   = torch.empty(upB.shape,   dtype=_cd, device=upB.device)\n"),
    # LoRA_MLP backward — remove old _bd cast block
    ("        # Force dtype consistency for all backward tensors\n        _bd = d_downA.dtype\n        h = h.to(_bd); dY = dY.to(_bd); df = df.to(_bd); de = de.to(_bd)\n        X = X.to(_bd)\n        downA = downA.to(_bd); downB = downB.to(_bd)\n        upA   = upA.to(_bd);   upB   = upB.to(_bd)\n        gateA = gateA.to(_bd); gateB = gateB.to(_bd)\n\n",
     "        h  = h.to(_cd);  df = df.to(_cd);  de = de.to(_cd)\n\n"),
    # LoRA_MLP backward — cast fast_dequantize outputs
    ("        upW = fast_dequantize(upW.t(), upW_quant)\n        dX = torch.matmul(df, upW.t(), out = X if ctx.inplace else None)\n",
     "        upW = fast_dequantize(upW.t(), upW_quant).to(_cd)\n        dX  = torch.matmul(df, upW.t(), out = X.to(_cd) if ctx.inplace else None)\n"),
    ("        gateW = fast_dequantize(gateW.t(), gateW_quant)\n",
     "        gateW = fast_dequantize(gateW.t(), gateW_quant).to(_cd)\n"),
    # LoRA_W backward — define _cd
    ("        dY = dY.reshape(-1, dY.shape[-1])  # Must be reshape\n        X = X.reshape(-1, X.shape[-1])  # Must be reshape\n        dtype = X.dtype\n",
     "        dY = dY.reshape(-1, dY.shape[-1])  # Must be reshape\n        X = X.reshape(-1, X.shape[-1])  # Must be reshape\n        _cd  = torch.get_autocast_gpu_dtype() if torch.is_autocast_enabled() else X.dtype\n        dtype = _cd\n        X    = X.to(_cd)\n        dY   = dY.to(_cd)\n"),
    # LoRA_W backward — empty_like
    ("        d_A = torch.empty_like(A)\n        d_B = torch.empty_like(B)\n",
     "        d_A = torch.empty(A.shape, dtype=_cd, device=A.device)\n        d_B = torch.empty(B.shape, dtype=_cd, device=B.device)\n"),
    # LoRA_W backward — cast fast_dequantize
    ("        W = fast_dequantize(W.t(), W_quant)\n        dX = dY @ W.t()\n",
     "        W  = fast_dequantize(W.t(), W_quant).to(_cd)\n        dX = dY @ W.t()\n"),
]

n = sum(1 for old, new in patches if old in src and src.__setattr__ or src.count(old) > 0)
applied = []
for old, new in patches:
    if old in src:
        src = src.replace(old, new, 1)
        applied.append(True)

with open(lora_path, "w") as f:
    f.write(src)

already = src.count("_cd   = torch.get_autocast_gpu_dtype")
print(f"fast_lora.py  ✓ {len(applied)} patches applied (already-patched lines skipped)")
print("\nAll patches done. Continue to next cell.")

In [ ]:
# ── 4. Download SFT adapter + dataset from HF Space ──────────────────────────
import os, shutil
from huggingface_hub import snapshot_download, hf_hub_download

HF_SPACE = "SidditaVarma/Built-different"
SFT_DIR  = "sft_adapter"
DATASET  = "dataset_train.jsonl"

if not os.path.isdir(SFT_DIR):
    snapshot_download(
        repo_id        = HF_SPACE,
        repo_type      = "space",
        allow_patterns = ["latest/blackstart-city-sft/**"],
        local_dir      = ".",
    )
    shutil.move("latest/blackstart-city-sft", SFT_DIR)
    print(f"Downloaded SFT adapter → {SFT_DIR}/")
else:
    print(f"Reusing existing {SFT_DIR}/")

if not os.path.isfile(DATASET):
    hf_hub_download(
        repo_id   = HF_SPACE,
        repo_type = "space",
        filename  = "latest/dataset_train.jsonl",
        local_dir = ".",
    )
    shutil.move("latest/dataset_train.jsonl", DATASET)
    print(f"Downloaded dataset → {DATASET}")
else:
    print(f"Reusing existing {DATASET}")

import subprocess
print(subprocess.run(["ls", "-lh", SFT_DIR], capture_output=True, text=True).stdout)

In [ ]:
# ── 5. Load base model + SFT adapter ─────────────────────────────────────────
import json, torch
from unsloth import FastLanguageModel, PatchFastRL, is_bfloat16_supported

PatchFastRL("GRPO", FastLanguageModel)

COMPUTE_DTYPE = torch.bfloat16 if is_bfloat16_supported() else torch.float16

cfg = json.load(open(f"{SFT_DIR}/adapter_config.json"))
BASE_MODEL = cfg["base_model_name_or_path"]
print("Base model :", BASE_MODEL)
print("Adapter r  :", cfg.get("r"))
print("LoRA alpha :", cfg.get("lora_alpha"))

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name             = BASE_MODEL,
    max_seq_length         = 4096,
    load_in_4bit           = True,
    fast_inference         = False,
    max_lora_rank          = 16,
    gpu_memory_utilization = 0.6,
)
model.warnings_issued = {}

model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    lora_alpha     = 32,
    use_gradient_checkpointing = True,
    random_state   = 3407,
)

model.load_adapter(SFT_DIR, adapter_name="sft_init", is_trainable=True, autocast_adapter_dtype=False)
model.set_adapter("sft_init")

# Cast all LoRA weights to compute dtype
for module in model.modules():
    for attr in ("lora_A", "lora_B"):
        d = getattr(module, attr, None)
        if isinstance(d, torch.nn.ModuleDict):
            for layer in d.values():
                if hasattr(layer, "weight") and layer.weight is not None:
                    layer.weight.data = layer.weight.data.to(COMPUTE_DTYPE)

print("Compute dtype :", COMPUTE_DTYPE)
print("Trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
# ── 6. Load, split & format dataset ──────────────────────────────────────────
from datasets import load_dataset

raw    = load_dataset("json", data_files=DATASET, split="train")
splits = raw.train_test_split(test_size=0.1, seed=42)

def to_chat(example):
    return {"prompt": [
        {"role": "system",  "content": "You are a city blackout restoration policy. "
                                       "Return exactly one valid JSON action object and nothing else."},
        {"role": "user",    "content": "Observation:\n" + example["prompt"]},
    ]}

train_dataset = splits["train"].map(to_chat, remove_columns=splits["train"].column_names)
test_dataset  = splits["test"].map(to_chat,  remove_columns=splits["test"].column_names)

print(f"Train: {len(train_dataset)} examples")
print(f"Test : {len(test_dataset)}  examples")

In [ ]:
# ── 7. Reward functions ───────────────────────────────────────────────────────
from blackstart_city.training.grpo_train import (
    format_reward_func,
    alignment_reward_func,
    action_quality_reward_func,
    constraint_reward_func,
    failure_context_reward_func,
)
print("Reward functions loaded.")

In [ ]:
# ── 8. GRPO config + trainer ──────────────────────────────────────────────────
from trl import GRPOConfig, GRPOTrainer

MAX_STEPS  = 500
OUTPUT_DIR = "grpo_output"

training_args = GRPOConfig(
    output_dir                  = OUTPUT_DIR,
    learning_rate               = 5e-6,
    lr_scheduler_type           = "cosine",
    max_steps                   = MAX_STEPS,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 2,
    num_generations             = 8,
    generation_batch_size       = 8,
    max_prompt_length           = 3500,
    max_completion_length       = 150,
    temperature                 = 0.9,
    bf16                        = is_bfloat16_supported(),
    fp16                        = not is_bfloat16_supported(),
    optim                       = "adamw_8bit",
    logging_steps               = 5,
    report_to                   = "none",
)

trainer = GRPOTrainer(
    model            = model,
    processing_class = tokenizer,
    reward_funcs     = [
        format_reward_func,
        alignment_reward_func,
        action_quality_reward_func,
        constraint_reward_func,
        failure_context_reward_func,
    ],
    reward_weights   = [0.2, 0.2, 0.2, 0.2, 0.2],
    args             = training_args,
    train_dataset    = train_dataset,
    eval_dataset     = test_dataset,
)
print("Trainer ready.")

In [ ]:
# ── 9. Train ──────────────────────────────────────────────────────────────────
trainer.train()

In [ ]:
# ── 10. Save + graphs ─────────────────────────────────────────────────────────
import os, matplotlib.pyplot as plt, numpy as np

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved → {OUTPUT_DIR}/")

def extract(key):
    return [(e["step"], e[key]) for e in trainer.state.log_history if key in e]

steps_rew, total_reward = zip(*extract("reward")) if extract("reward") else ([], [])
steps_kl,  kl_vals      = zip(*extract("kl"))     if extract("kl")     else ([], [])
steps_los, loss_vals    = zip(*extract("loss"))   if extract("loss")   else ([], [])

reward_keys = [
    ("rewards/format_reward_func/mean",          "Format",        "#FF00FF"),
    ("rewards/alignment_reward_func/mean",       "Alignment",     "#00CCFF"),
    ("rewards/action_quality_reward_func/mean",  "Tact. Quality", "#00FF66"),
    ("rewards/constraint_reward_func/mean",      "Constraint",    "#FFD700"),
    ("rewards/failure_context_reward_func/mean", "Fail. Avoid",   "#FF6B6B"),
]

plt.style.use("dark_background")
fig, axes = plt.subplots(1, 3, figsize=(24, 6), facecolor="#121212")
fig.suptitle("GRPO Training — SFT Warm-Start", color="#00FFCC", fontsize=15, fontweight="bold")

ax = axes[0]
ax.set_facecolor("#1e1e1e")
for key, label, color in reward_keys:
    data = extract(key)
    if data:
        xs, ys = zip(*data)
        ax.plot(xs, ys, color=color, linewidth=1.5, alpha=0.85, label=label)
ax.set_title("Per-Signal Rewards", color="#00FFCC", fontweight="bold")
ax.set_xlabel("Step"); ax.set_ylabel("Reward")
ax.legend(facecolor="#1e1e1e", edgecolor="#444", labelcolor="white", fontsize=8)
ax.grid(True, linestyle="--", alpha=0.15)

ax2 = axes[1]
ax2.set_facecolor("#1e1e1e")
if total_reward:
    ax2.plot(steps_rew, total_reward, color="#00FFCC", linewidth=2)
ax2.set_xlabel("Step"); ax2.set_ylabel("Total Reward", color="#00FFCC")
ax2.tick_params(axis="y", labelcolor="#00FFCC")
ax2.set_title("Total Reward & KL", color="#00FFCC", fontweight="bold")
ax2.grid(True, linestyle="--", alpha=0.15)
if kl_vals:
    ax2b = ax2.twinx()
    ax2b.plot(steps_kl, kl_vals, color="#FF6B6B", linewidth=1.5, linestyle="--", alpha=0.7)
    ax2b.set_ylabel("KL Divergence", color="#FF6B6B")
    ax2b.tick_params(axis="y", labelcolor="#FF6B6B")

ax3 = axes[2]
ax3.set_facecolor("#1e1e1e")
if loss_vals:
    ax3.plot(steps_los, loss_vals, color="#FFD700", linewidth=1.5)
ax3.set_title("Policy Loss", color="#00FFCC", fontweight="bold")
ax3.set_xlabel("Step"); ax3.set_ylabel("Loss")
ax3.grid(True, linestyle="--", alpha=0.15)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/training_summary.png", dpi=150, bbox_inches="tight", facecolor="#121212")
plt.show()

if total_reward:
    print(f"Steps run    : {max(steps_rew)}")
    print(f"Reward start : {np.mean(total_reward[:5]):.4f}")
    print(f"Reward end   : {np.mean(total_reward[-5:]):.4f}")
    print(f"Reward change: {np.mean(total_reward[-5:]) - np.mean(total_reward[:5]):+.4f}")

In [ ]:
# ── 11. Push to HF Space ──────────────────────────────────────────────────────
import os
from huggingface_hub import HfApi, login

HF_TOKEN = ""   # paste your write token from huggingface.co/settings/tokens

if HF_TOKEN:
    login(token=HF_TOKEN)
    api = HfApi()

    upload_files = [
        "adapter_model.safetensors",
        "adapter_config.json",
        "tokenizer.json",
        "tokenizer_config.json",
        "special_tokens_map.json",
        "chat_template.jinja",
        "training_summary.png",
    ]

    for filename in upload_files:
        filepath = f"{OUTPUT_DIR}/{filename}"
        if os.path.exists(filepath):
            api.upload_file(
                path_or_fileobj = filepath,
                path_in_repo    = f"latest/blackstart-city-grpo/{filename}",
                repo_id         = "SidditaVarma/Built-different",
                repo_type       = "space",
                commit_message  = f"Add GRPO: {filename}",
            )
            print(f"Uploaded {filename}")

    print("\nDone: https://huggingface.co/spaces/SidditaVarma/Built-different/tree/main/latest/blackstart-city-grpo")
else:
    print("Set HF_TOKEN to upload.")